## 일반 글쓰기까지 가능 코드

In [3]:
from selenium import webdriver 
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
import pyperclip
import time
import random 

print("=" *80)
print(" 개인프로젝트 네이버 블로그 자동 글쓰기 (우회 공격 + 팝업제거 최종본)")
print("=" *80)
print("\n")

# -------------------------------------------------------------
# 1. 자동 로그인을 위한 네이버 ID/PW 
# -------------------------------------------------------------
v_id = input('🔑 네이버 로그인 ID를 입력하세요: ')
v_passwd = input('🔑 네이버 로그인 비밀번호를 입력하세요: ')

blog_title = "자동차에서 에어컨 냄새가 날 때 해결방법"
blog_content = """안녕하세요!
이 글은 파이썬 셀레니움을 활용하여 사람처럼 타이핑해서 작성한 테스트 블로그 글입니다.

추후 쿠팡파트너스 광고글 및 일상 정보를 자동으로 적을 수 있습니다.
봇 탐지 우회를 위해 한 글자씩 자연스럽게 타이핑되고 있습니다.

읽어주셔서 감사합니다!"""

print("\n🚀 블로그 포스팅 작성을 시작합니다. 브라우저가 열리면 잠시 지켜봐주세요!")

options = Options()
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=options) 
driver.maximize_window()

try:
    # -------------------------------------------------------------
    # 2 & 3. 🎯 핵심: 블로그부터 찌르고 리다이렉트 된 로그인 창에서 우회 성공하기
    # -------------------------------------------------------------
    print(">> 블로그 글쓰기 페이지로 먼저 다이렉트 접근 시도...")
    # 로그인 없이 바로 글쓰기 창으로 들어갑니다. (자동으로 네이버 로그인 창으로 튕김)
    driver.get(f"https://blog.naver.com/{v_id}?Redirect=Write")
    time.sleep(3)

    print(">> 리다이렉트 된 창에서 로그인을 시도합니다.")
    actions = ActionChains(driver)

    # 아이디 클릭 후 클립보드(Ctrl+V) 붙여넣기 방식 사용 (가장 안전)
    id_element = driver.find_element(By.NAME, 'id')
    id_element.click()
    pyperclip.copy(v_id) 
    actions.key_down(Keys.CONTROL).send_keys('v').key_up(Keys.CONTROL).perform()
    time.sleep(1)

    # 비밀번호 클릭 후 클립보드(Ctrl+V) 붙여넣기
    pw_element = driver.find_element(By.NAME, 'pw')
    pw_element.click()
    pyperclip.copy(v_passwd) 
    actions.key_down(Keys.CONTROL).send_keys('v').key_up(Keys.CONTROL).perform()
    time.sleep(1)

    # 로그인 버튼 클릭
    driver.find_element(By.ID, 'log.login').click()  
    print(">> 로그인 클릭 완료! 블로그 에디터 창으로 자동 복귀를 기다립니다...")
    
    # -------------------------------------------------------------
    # 4. 로그인 성공 후 자동 복귀된 블로그 글쓰기 대기
    # -------------------------------------------------------------
    # 별도로 driver.get(블로그)를 할 필요가 없습니다. 네이버가 알아서 돌려보내 줍니다!
    time.sleep(7)  
    
    wait = WebDriverWait(driver, 15)
    
    # [중요] 스마트에디터는 'mainFrame'이라는 iframe 안에 존재합니다.
    driver.switch_to.frame("mainFrame")
    
    # -------------------------------------------------------------
    # 5. 블로그 에디터 팝업 처리 (이어쓰기, 도움말 창 닫기)
    # -------------------------------------------------------------
    try:
        time.sleep(1.5)
        cancel_btn = driver.find_element(By.CSS_SELECTOR, 'button.se-popup-button-cancel')
        if cancel_btn.is_displayed():
            cancel_btn.click()
            time.sleep(1)
            print(">> [성공] '이어서 작성하시겠습니까?' 팝업 취소 완료")
    except:
        pass

    try:
        time.sleep(1)
        help_close_btn = driver.find_element(By.CSS_SELECTOR, 'button.se-help-panel-close-button')
        if help_close_btn.is_displayed():
            help_close_btn.click()
            time.sleep(1)
            print(">> [성공] 우측 상단 '도움말 툴팁 가이드' 창 닫기 완료")
    except:
        pass

    # -------------------------------------------------------------
    # 6. 블로그 제목 입력 (사람처럼 타이핑)
    # -------------------------------------------------------------
    print(">> 제목 입력 시작")
    title_field = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "div.se-documentTitle")))
    title_field.click()
    time.sleep(1)
    
    for char in blog_title:
        actions.send_keys(char).perform()
        time.sleep(random.uniform(0.05, 0.15))
        
    time.sleep(1)

    # -------------------------------------------------------------
    # 7. 블로그 본문 입력
    # -------------------------------------------------------------
    print(">> 본문 입력 시작")
    actions.send_keys(Keys.ENTER).perform() 
    time.sleep(1)
    
    for line in blog_content.split('\n'):
        for char in line:
            actions.send_keys(char).perform()
            time.sleep(random.uniform(0.02, 0.08)) 
        
        actions.send_keys(Keys.ENTER).perform() 
        time.sleep(random.uniform(0.3, 0.6))
        
    time.sleep(2)
    print(">> 제목/본문 작성 완료! 발행 과정을 시작합니다.")

    # -------------------------------------------------------------
    # 8. 발행하기 클릭 (이중 클릭 포함)
    # -------------------------------------------------------------
    print(">> 1차 발행 버튼(설정 열기) 클릭 시도 중...")
    first_publish_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button[class*='publish_btn'], button[data-click-area='tpb.publish']")))
    first_publish_btn.click()
    print(">> 1차 발행 버튼 클릭 완료. 설정 패널 렌더링을 기다립니다.")
    
    time.sleep(3) 
    
    print(">> 2차 최종 발행 버튼 찾는 중...")
    final_publish_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button[class*='confirm_btn'], button[data-testid='seOnePublishBtn']")))
    final_publish_btn.click()
    print(">> 🚀 최종 승인 자동화 성공! 발행이 무사히 완료되었습니다!")
        
    time.sleep(7)

except Exception as e:
    print("\n[에러 발생] 진행 도중 문제가 생겼습니다/찾을 수 없는 요소가 있었습니다:", e)


 개인프로젝트 네이버 블로그 자동 글쓰기 (우회 공격 + 팝업제거 최종본)



🚀 블로그 포스팅 작성을 시작합니다. 브라우저가 열리면 잠시 지켜봐주세요!
>> 블로그 글쓰기 페이지로 먼저 다이렉트 접근 시도...
>> 리다이렉트 된 창에서 로그인을 시도합니다.
>> 로그인 클릭 완료! 블로그 에디터 창으로 자동 복귀를 기다립니다...

[에러 발생] 진행 도중 문제가 생겼습니다/찾을 수 없는 요소가 있었습니다: Message: mainFrame



## 파트너스 csv파일 입력 코드

In [1]:
from selenium import webdriver 
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
import pyperclip
import time
import random 
import csv 

print("=" *80)
print(" 개인프로젝트 쿠팡 파트너스 블로그 자동 글쓰기 (절대경로 반영 최종본)")
print("=" *80)
print("\n")

# -------------------------------------------------------------
# 0. 쿠팡 파트너스 CSV 데이터 불러오기 및 글 템플릿 세팅
# -------------------------------------------------------------
# 원본 파일이 있는 절대경로를 명시하여 폴더 위치가 달라도 무조건 찾을 수 있게 합니다.
# (r을 앞에 붙여 역슬래시(\)로 인한 이스케이프 이벤트를 방지합니다)
csv_file_path = r'c:\Users\itwill\workspace\네이버 자동화 ing\naver 자동화\products_db.csv' 

try:
    with open(csv_file_path, 'r', encoding='cp949') as f:
        reader = csv.DictReader(f)
        products = list(reader)
except UnicodeDecodeError:
    with open(csv_file_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        products = list(reader)

# 랜덤으로 상품 하나 선택
target = random.choice(products)
p_name = target['상품명']
p_keyword = target['키워드']
p_link = target['쿠팡링크']

# 블로그 제목과 내용 자동 생성 (광고 문구 최상단 탑재)
blog_title = f"[{p_keyword}] 가성비 최고! '{p_name}' 강력 추천"
blog_content = f"""🚨 본 포스팅은 쿠팡 파트너스 활동의 일환으로, 이에 따른 일정액의 수수료를 제공받습니다.

안녕하세요! 😊
오늘은 [{p_keyword}] 관련해서 좋은 리뷰와 평가를 받고 있는 꿀템을 소개해 드리려고 합니다.

정말 많은 분들이 찾으시는 제품이라 꼼꼼히 알아보고 준비했습니다.
제가 오늘 강력하게 추천드리는 제품은 바로!
👉 {p_name} 👈
입니다.

자세한 가격 정보나 다른 분들의 생생한 후기를 읽어보고 싶으시다면, 아래 링크를 통해서 바로 확인하실 수 있습니다!

✅ 제품 상세정보 및 구매 링크 바로가기:
{p_link}

오늘 하루도 즐겁고 행복하게 보내시길 바랍니다.
항상 감사합니다!"""

# -------------------------------------------------------------
# 1. 자동 로그인을 위한 네이버 ID/PW 
# -------------------------------------------------------------
v_id = input('🔑 네이버 로그인 ID를 입력하세요: ')
v_passwd = input('🔑 네이버 로그인 비밀번호를 입력하세요: ')

print(f"\n🚀 '{p_name}' 쿠팡 파트너스 자동 포스팅을 시작합니다!")

# -------------------------------------------------------------
# 2. 크롬 드라이버 셋팅
# -------------------------------------------------------------
options = Options()
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=options) 
driver.maximize_window()

try:
    # -------------------------------------------------------------
    # 3. 우회 공격 방식 (블로그부터 찌르고 리다이렉트 된 로그인 창에서 로그인)
    # -------------------------------------------------------------
    driver.get(f"https://blog.naver.com/{v_id}?Redirect=Write")
    time.sleep(3)

    print(">> 리다이렉트 된 창에서 로그인을 시도합니다.")
    actions = ActionChains(driver)

    # 아이디 입력 (클립보드 방식)
    id_element = driver.find_element(By.NAME, 'id')
    id_element.click()
    pyperclip.copy(v_id) 
    actions.key_down(Keys.CONTROL).send_keys('v').key_up(Keys.CONTROL).perform()
    time.sleep(1)

    # 비밀번호 입력
    pw_element = driver.find_element(By.NAME, 'pw')
    pw_element.click()
    pyperclip.copy(v_passwd) 
    actions.key_down(Keys.CONTROL).send_keys('v').key_up(Keys.CONTROL).perform()
    time.sleep(1)

    driver.find_element(By.ID, 'log.login').click()  
    print(">> 로그인 통과! 블로그 에디터 창으로 자동 복귀 대기중...")
    
    # -------------------------------------------------------------
    # 4 & 5. 로그인 성공 후 에디터 진입 및 팝업 처리
    # -------------------------------------------------------------
    time.sleep(7)  
    
    wait = WebDriverWait(driver, 10)
    driver.switch_to.frame("mainFrame")
    
    # 임시저장 취소 창 닫기
    try:
        time.sleep(1.5)
        cancel_btn = driver.find_element(By.CSS_SELECTOR, 'button.se-popup-button-cancel')
        if cancel_btn.is_displayed():
            cancel_btn.click()
            time.sleep(1)
            print(">> [성공] '이어서 작성하시겠습니까?' 팝업 취소")
    except:
        pass

    # 우측 상단 도움말 창 닫기
    try:
        time.sleep(1)
        help_close_btn = driver.find_element(By.CSS_SELECTOR, 'button.se-help-panel-close-button')
        if help_close_btn.is_displayed():
            help_close_btn.click()
            time.sleep(1)
            print(">> [성공] 우측 상단 '도움말' 닫기")
    except:
        pass

    # -------------------------------------------------------------
    # 6. 블로그 제목 입력
    # -------------------------------------------------------------
    print(">> 제목 타이핑 시작")
    title_field = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "div.se-documentTitle")))
    title_field.click()
    time.sleep(1)
    
    for char in blog_title:
        actions.send_keys(char).perform()
        time.sleep(random.uniform(0.05, 0.15))
        
    time.sleep(1)

    # -------------------------------------------------------------
    # 7. 블로그 본문 입력
    # -------------------------------------------------------------
    print(">> 본문 타이핑 시작 (내용이 길어 빠른 속도로 타이핑합니다)")
    actions.send_keys(Keys.ENTER).perform() 
    time.sleep(1)
    
    for line in blog_content.split('\n'):
        for char in line:
            actions.send_keys(char).perform()
            time.sleep(random.uniform(0.01, 0.04)) 
        
        actions.send_keys(Keys.ENTER).perform() 
        time.sleep(random.uniform(0.3, 0.6))
        
    time.sleep(2)
    print(">> 쿠팡 파트너스 홍보글 타이핑 완료!")

    # -------------------------------------------------------------
    # 8. 발행하기 클릭 (이중 클릭)
    # -------------------------------------------------------------
    print(">> 1차 발행 버튼 클릭...")
    first_publish_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button[class*='publish_btn'], button[data-click-area='tpb.publish']")))
    first_publish_btn.click()
    
    time.sleep(3) 
    
    print(">> 2차 최종 발행 버튼 찾는 중...")
    final_publish_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "button[class*='confirm_btn'], button[data-testid='seOnePublishBtn']")))
    final_publish_btn.click()
    print(">> 🚀 쿠팡 파트너스 수익 자동화! 글 발행이 무사히 완료되었습니다!")
        
    time.sleep(7)

except Exception as e:
    print("\n[에러 발생] 진행 도중 문제가 생겼습니다:", e)


 개인프로젝트 쿠팡 파트너스 블로그 자동 글쓰기 (절대경로 반영 최종본)



🚀 '캐리어 벽걸이 에어컨' 쿠팡 파트너스 자동 포스팅을 시작합니다!
>> 리다이렉트 된 창에서 로그인을 시도합니다.
>> 로그인 통과! 블로그 에디터 창으로 자동 복귀 대기중...

[에러 발생] 진행 도중 문제가 생겼습니다: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=146.0.7680.165); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff6c8ff29c5+2ed785]
	chromedriver!GetHandleVerifier [0x7ff6c8d1a0d0+14e90]
	chromedriver!(No symbol) [0x7ff6c8a7db2d]
	chromedriver!(No symbol) [0x7ff6c8a68ca2]
	chromedriver!(No symbol) [0x7ff6c8a8f416]
	chromedriver!(No symbol) [0x7ff6c8b070e0]
	chromedriver!(No symbol) [0x7ff6c8b23482]
	chromedriver!(No symbol) [0x7ff6c8ac9298]
	chromedriver!(No symbol) [0x7ff6c8aca183]
	chromedriver!GetHandleVerifier [0x7ff6c901de0d+318bcd]
	chromedriver!GetHandleVerifier